In [0]:
# ---- CELL 1: config ---------------------------------------------------
CATALOG = "workspace"
SCHEMA  = "cafe_karan"

SILVER_TABLE     = f"{CATALOG}.{SCHEMA}.silver_cafe_sales"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_cafe_sales_quarantine"
BRONZE_TABLE     = f"{CATALOG}.{SCHEMA}.bronze_cafe_sales"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table(SILVER_TABLE)
print("Silver rows:", silver.count())   # 9977

# COMMAND ----------

# ---- CELL 2: headline KPIs (one row) ----------------------------------
kpi_summary = silver.select(
    F.round(F.sum("total_spent"), 2).alias("total_revenue"),
    F.count("*").alias("total_transactions"),
    F.round(F.sum("quantity"), 0).alias("total_units_sold"),
    F.round(F.avg("total_spent"), 2).alias("avg_order_value"),
    F.countDistinct("item").alias("distinct_items_sold"),
    F.min("transaction_date").alias("first_transaction"),
    F.max("transaction_date").alias("last_transaction"),
)

kpi_summary.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA}.gold_kpi_summary")

display(kpi_summary)

# COMMAND ----------

# ---- CELL 3: one helper, reused for every breakdown -------------------
# Three things every breakdown must do:
#   1. coalesce NULL -> "Unknown"  so nothing vanishes from the totals
#   2. compute a % share using a window over the whole result
#   3. sort by revenue so the chart is readable without touching it
def revenue_by(df, dimension, alias=None):
    alias = alias or dimension
    total = F.sum("revenue").over(Window.partitionBy())   # grand total on every row
    return (
        df
        .withColumn(alias, F.coalesce(F.col(dimension), F.lit("Unknown")))
        .groupBy(alias)
        .agg(
            F.round(F.sum("total_spent"), 2).alias("revenue"),
            F.count("*").alias("transactions"),
            F.round(F.sum("quantity"), 0).alias("units"),
            F.round(F.avg("total_spent"), 2).alias("avg_order_value"),
        )
        .withColumn("revenue_share_pct", F.round(F.col("revenue") / total * 100, 2))
        .orderBy(F.col("revenue").desc())
    )

# COMMAND ----------

# ---- CELL 4/5/6: the three dimension breakdowns -----------------------
for dim, table in [
    ("item",           "gold_revenue_by_item"),
    ("payment_method", "gold_revenue_by_payment_method"),
    ("location",       "gold_revenue_by_location"),
]:
    out = revenue_by(silver, dim)
    out.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
    print(table)
    display(out)

# COMMAND ----------

# ---- CELL 7: monthly trend --------------------------------------------
# date_format(NULL) is NULL, so the same Unknown-bucket rule applies.
monthly = (
    silver
    .withColumn("month", F.coalesce(F.date_format("transaction_date", "yyyy-MM"),
                                    F.lit("Unknown")))
    .groupBy("month")
    .agg(
        F.round(F.sum("total_spent"), 2).alias("revenue"),
        F.count("*").alias("transactions"),
        F.round(F.avg("total_spent"), 2).alias("avg_order_value"),
    )
    .orderBy("month")
)

monthly.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA}.gold_revenue_by_month")

display(monthly)

# COMMAND ----------

# ---- CELL 8: data quality KPIs ----------------------------------------
# These belong in gold too. "How much of this can we trust?" is a
# business question, not an engineering detail.
bronze_n     = spark.table(BRONZE_TABLE).count()
quarantine_n = spark.table(QUARANTINE_TABLE).count()

dq = silver.select(
    F.lit(bronze_n).alias("rows_ingested"),
    F.count("*").alias("rows_usable"),
    F.lit(quarantine_n).alias("rows_quarantined"),
    F.round(F.count("*") / F.lit(bronze_n) * 100, 2).alias("usable_pct"),
    F.sum(F.col("_total_was_missing").cast("int")).alias("revenue_values_recovered"),
    F.sum(F.col("_price_was_missing").cast("int")).alias("price_values_recovered"),
    F.sum(F.col("_quantity_was_missing").cast("int")).alias("qty_values_recovered"),
    F.sum(F.col("item").isNull().cast("int")).alias("item_unknown"),
    F.round(F.sum(F.col("payment_method").isNull().cast("int")) / F.count("*") * 100, 2)
        .alias("payment_unknown_pct"),
    F.round(F.sum(F.col("location").isNull().cast("int")) / F.count("*") * 100, 2)
        .alias("location_unknown_pct"),
)

dq.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True).saveAsTable(f"{CATALOG}.{SCHEMA}.gold_data_quality")

display(dq)

# COMMAND ----------

# ---- CELL 9: reconciliation check -------------------------------------
# Every breakdown must sum back to the headline. If one doesn't, a NULL
# dimension got dropped somewhere and a KPI is understated.
headline = spark.table(f"{CATALOG}.{SCHEMA}.gold_kpi_summary") \
                .collect()[0]["total_revenue"]

for t in ["gold_revenue_by_item", "gold_revenue_by_payment_method",
          "gold_revenue_by_location", "gold_revenue_by_month"]:
    s = spark.table(f"{CATALOG}.{SCHEMA}.{t}").agg(F.sum("revenue")).collect()[0][0]
    print(f"{t:<35} {s:>12,.2f}   {'OK' if abs(s - headline) < 0.01 else 'MISMATCH'}")